## Optimize CNMF

In [ ]:
import os
import glob
from copy import deepcopy

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import mesmerize_core as mc
import mesmerize_viz
import fastplotlib as fpl

from mesmerize_core.caiman_extensions.cnmf import cnmf_cache

from pathlib import Path

if os.name == "nt":
    # disable the cache on windows, this will be automatic in a future version
    cnmf_cache.set_maxsize(0)

pd.options.display.max_colwidth = 120

## Set paths

In [ ]:
# data path
data_path = Path('D:/Data/2P/C57_O1M2/10022023/TSeries-10022023-1300-001')
print(f"Load data from {data_path}.")

# Set output path and assign it as raw data path (required by df.caiman.add_item below)
export_path = Path('H:/Analysis/2P/C57_O1M2/10022023/run1/mesmerize/')
mc.set_parent_raw_data_path(export_path) 

# Create export_path if it doesn't exist
if not os.path.exists(export_path):
    os.makedirs(export_path)

# movie path
movie_path = Path.joinpath(export_path, 'cat_tiff_bt.tiff')

# batch path
batch_path = Path.joinpath(export_path, 'batch.pickle')

print(f"Export data to {export_path}.") 
print(f"Movie path: {movie_path}.") 
print(f"Batch path: {batch_path}.") 

### Load a batch

In [ ]:
df = mc.load_batch(batch_path)
df

### Remove unneeded df rows

In [ ]:
# make a list of rows we want to keep using the uuids
rows_keep = [df.iloc[1].uuid] #, df.iloc[1].uuid, df.iloc[2].uuid
rows_keep

In [ ]:
for i, row in df.iterrows():
    if row.uuid not in rows_keep:
         df.caiman.remove_item(row.uuid)
df

### Replace mmap
If a clipped memmap file exists, delete the original memmap file and rename the clipped memmap file to the original name

In [ ]:

outputs_dict = df.iloc[0]["outputs"]
mc_memmapped_fname = Path.joinpath(export_path, outputs_dict['mcorr-output-path'])
print(f"Original memmap file: {mc_memmapped_fname}.")

# Check if a clipped memmap file exists (i.e. same name as original but with "clipped_" appended to the beginning)
clipped_mmap_path = Path.joinpath(mc_memmapped_fname.parent,
                                  f"clipped_{mc_memmapped_fname.name}")
# check if the clipped memmap file exists
if clipped_mmap_path.exists():
    print(f"Clipped memmap file exists: {clipped_mmap_path}.")
    # if it does, delete the original memmap file
    mc_memmapped_fname.unlink()
    # and rename the clipped memmap file to the original name
    clipped_mmap_path.rename(mc_memmapped_fname)
    print(f"Clipped memmap file renamed to {mc_memmapped_fname}.")

### CNMF
Perform CNMF using the mcorr output.
Similar to mcorr, put the CNMF params within the `main` key. The `refit` key will perform a second iteration, as shown in the `caiman` `demo_pipeline.ipynb` notebook.

### Set params

In [ ]:
# some params for CNMF
params_cnmf =\
{
    'main': # indicates that these are the "main" params for the CNMF algo
        {
            'fr': 20, # framerate, very important!
            'p': 1,
            'nb': 2,
            'merge_thr': 0.85,
            'rf': 20,
            'stride': 10, # "stride" for cnmf, "strides" for mcorr
            'K': 10,
            'gSig': [5, 5],
            'ssub': 1,
            'tsub': 1,
            'method_init': 'greedy_roi',
            'min_SNR': 3.0,
            'SNR_lowest':  1.0,
            'rval_thr': 0.8,
            'rval_lowest': 0.2,
            'use_cnn': True,
            'min_cnn_thr': 0.9,
            'cnn_lowest': 0.2,
            'decay_time': 0.15,
        },
    'refit': True, # If `True`, run a second iteration of CNMF
}
# Create second set of parameters
# new_params_cnmf = deepcopy(params_cnmf)
# new_params_cnmf["main"]["gSig"] = [4, 4]

### Add CNMF item

You can provide the mcorr item row to `input_movie_path` and it will resolve the path of the input movie from the entry in the DataFrame.

In [ ]:
good_mcorr_index = 0
 
# add batch items
df.caiman.add_item(
    algo='cnmf', # algo is cnmf
    input_movie_path=df.iloc[good_mcorr_index],  # use mcorr output from a completed batch item
    params=params_cnmf,
    item_name=df.iloc[good_mcorr_index]["item_name"], # use the same item name
)
# add a batch item
# df.caiman.add_item(
#     algo='cnmf', # algo is cnmf
#     input_movie_path=df.iloc[good_mcorr_index],  # use mcorr output from a completed batch item
#     params=new_params_cnmf,
#     item_name=df.iloc[good_mcorr_index]["item_name"], # use the same item name
# )

See the cnmf item at the bottom of the dataframe

In [ ]:
df

### Run CNMF

In [ ]:
# Run rows that haven't been run yet
for i, row in df.iterrows():
    if row["outputs"] is not None: # item has already been run
        continue # skip
        
    process = row.caiman.run()
    
    # on Windows you MUST reload the batch dataframe after every iteration because it uses the `local` backend.
    # this is unnecessary on Linux & Mac
    # "DummyProcess" is used for local backend so this is automatic
    if process.__class__.__name__ == "DummyProcess":
        df = df.caiman.reload_from_disk()

### Reload dataframe

In [ ]:
df = df.caiman.reload_from_disk()
df

### Visualize with Fastplotlib and LazyArrays

In [ ]:
cnmf_index = 1
rcm = df.iloc[cnmf_index].cnmf.get_rcm()
rcb = df.iloc[cnmf_index].cnmf.get_rcb()
residuals = df.iloc[cnmf_index].cnmf.get_residuals()
input_movie = df.iloc[cnmf_index].caiman.get_input_movie()

In [ ]:
iw_rcm = fpl.ImageWidget(
    data=[input_movie, rcm, rcb, residuals], 
    grid_plot_kwargs={"size": (800, 600)}, 
    cmap="gnuplot2"
)
iw_rcm.show()

In [ ]:
iw_rcm.close()

### Visualize everything with `mesmerize-viz`

In [ ]:
# %gui qt

In [ ]:
viz_simple = df.cnmf.viz(
    # image_data_options=["input", "rcm"],
    image_data_options=["max"],
)
viz_simple.show(sidecar=True)
# viz_simple.show()
viz_simple.image_widget.cmap = "gray"

In [ ]:
viz_simple.close()

### Save parameters to disk as a json file

In [ ]:
import json
params_path = Path.joinpath(export_path, 'caiman_params.json')

caiman_params = {
    'data_path': str(data_path),
    'export_path': str(export_path),
    'movie_name': str(movie_path.name)
}
caiman_mcorr_params = df.iloc[0]["params"]['main']
caiman_mcorr_params['timestamp_mcorr'] = df.iloc[0]["ran_time"]
caiman_cnmf_params = params_cnmf['main'] | df.iloc[-1]["params"]['main']
caiman_cnmf_params['timestamp_cnmf'] = df.iloc[-1]["ran_time"]
caiman_params = caiman_params | caiman_mcorr_params | caiman_cnmf_params

with open(params_path, 'w') as f:
    json.dump(caiman_params, f, indent=4) #sort_keys=True)

### Save to Matlab

In [ ]:
# View the content of estimates
# cnmf_obj.estimates?

In [ ]:
# Prepare outputs for saving
cnmf_obj = df.iloc[-1].cnmf.get_output()

# Keep only accepted components
cnmf_obj.estimates.select_components(use_object=True)

# Compute dF/F
if cnmf_obj.estimates.F_dff is None:
    print('Calculating estimates.F_dff')
    cnmf_obj.estimates.detrend_df_f(quantileMin=8, 
                                      frames_window=400,
                                      use_residuals=False);  # use denoised data

In [ ]:
# save results for matlab
from scipy import io
# Lists must be converted to arrays, otherwise Matlab loads them as chars
io.savemat(Path.joinpath(export_path, 'results_caiman.mat'), mdict={
                                                 'mean_map_motion_corrected': df.iloc[0].caiman.get_projection("mean"),
                                                 'spatial_components': cnmf_obj.estimates.A,
                                                 'temporal_components': cnmf_obj.estimates.C,
                                                 'background_spatial_component': cnmf_obj.estimates.b,
                                                 'background_temporal_component': cnmf_obj.estimates.f,
                                                 'df_wo_bckgrnd': cnmf_obj.estimates.F_dff,
                                                 'deconv_spk': cnmf_obj.estimates.S,
                                                 'SNR_comp': cnmf_obj.estimates.SNR_comp,
                                                 'baseline': np.array(cnmf_obj.estimates.bl, dtype=np.float32),
                                                 'noise': np.array(cnmf_obj.estimates.neurons_sn, dtype=np.float32)})